In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

def bacon_watts_method(time, capacity):
    """
    Detect the turning point (knee point) in the capacity curve using the Bacon-Watts method.
    
    Parameters:
    - time: Original time data (e.g., cycle number)
    - capacity: Original capacity data

    Returns:
    - best_split_index: Index of the knee point in the data
    - jump_point_time: Original time value corresponding to the knee point
    - jump_point_capacity: Original capacity value corresponding to the knee point
    """
    n = len(time)
    best_split_index = 0
    min_total_error = float('inf')

    # Traverse all possible split points
    for split in range(1, n - 1):
        # Split the data into left and right segments
        time_left, capacity_left = time[:split], capacity[:split]
        time_right, capacity_right = time[split:], capacity[split:]
        
        # Fit linear models to the left and right segments
        model_left = LinearRegression().fit(time_left.reshape(-1, 1), capacity_left)
        model_right = LinearRegression().fit(time_right.reshape(-1, 1), capacity_right)
        
        # Calculate prediction errors for both segments
        predicted_left = model_left.predict(time_left.reshape(-1, 1))
        predicted_right = model_right.predict(time_right.reshape(-1, 1))
        error_left = np.sum((capacity_left - predicted_left) ** 2)
        error_right = np.sum((capacity_right - predicted_right) ** 2)
        
        # Compute total error
        total_error = error_left + error_right

        # Update best split point if current split yields a smaller error
        if total_error < min_total_error:
            min_total_error = total_error
            best_split_index = split

    # Retrieve the original values at the knee point
    jump_point_time = time[best_split_index]
    jump_point_capacity = capacity[best_split_index]

    return best_split_index, jump_point_time, jump_point_capacity


In [2]:
def remove_outliers_3sigma(capacity, cycle, k=3.0):
    """
    Detect outliers using the 3-sigma rule and repair them via linear interpolation.

    Parameters
    ----------
    capacity : array-like
        Capacity values (e.g., discharge capacity or SOH).
    cycle : array-like
        Cycle indices corresponding to the capacity values.
    k : float, optional
        Sigma threshold for outlier detection (default is 3.0).

    Returns
    -------
    capacity_clean : ndarray
        Capacity sequence after outlier removal and interpolation.
    outliers : ndarray (bool)
        Boolean mask indicating detected outlier positions.
    """
    capacity = capacity.astype(float)

    mean = np.mean(capacity)
    std = np.std(capacity)

    # Identify outliers based on the k-sigma criterion
    outliers = (capacity < mean - k * std) | (capacity > mean + k * std)

    # Replace outliers with NaN for interpolation
    capacity_clean = capacity.copy()
    capacity_clean[outliers] = np.nan

    # Repair outliers using linear interpolation
    capacity_clean = (
        pd.Series(capacity_clean)
        .interpolate(method="linear")
        .bfill()
        .ffill()
        .values
    )

    return capacity_clean, outliers


In [3]:
import os
import pandas as pd

# =========================
# Path configuration
# =========================
data_dir = r"./.../knee"

# =========================
# Containers for results
# =========================
tot_cap, tot_eol, tot_tp = [], [], []

# =========================
# Traverse all CSV files in the knee directory
# =========================
for fname in os.listdir(data_dir):
    file_path = os.path.join(data_dir, fname)

    if not fname.lower().endswith(".csv"):
        continue
    if not os.path.isfile(file_path):
        continue

    battery_name = fname.replace("_Q_cycle.csv", "")

    # =========================
    # Load cycle-level capacity data
    # =========================
    df = pd.read_csv(file_path)

    cycle = df["cycle number"].values
    capacity = df["Q discharge/mA.h"].values.astype(float)

    # =========================
    # Outlier detection and repair (3-sigma + interpolation)
    # =========================
    capacity_clean, outlier_mask = remove_outliers_3sigma(
        capacity, cycle
    )

    # =========================
    # Normalization (SOH w.r.t. initial capacity)
    # =========================
    capacity_norm = capacity_clean / capacity_clean[0]

    # =========================
    # Knee point detection using Bacon–Watts method
    # =========================
    best_split_index, knee_cycle, knee_capacity = bacon_watts_method(
        cycle, capacity_norm
    )

    # =========================
    # Store results
    # =========================
    tot_cap.append(capacity_norm)      # normalized full curve (SOH)
    # tot_tp.append(knee_cycle)           # knee cycle
    tot_eol.append(len(capacity_norm))  # lifetime length
    tot_tp.append(best_split_index)


In [4]:
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
import numpy as np
import matplotlib.pyplot as plt

# Nonlinear degradation model: Q_loss = D1 * sqrt(x) + D2 * x^(3/2)
def func(x, D1, D2):
    return D1 * np.sqrt(x) + D2 * np.power(x, 1.5)

# R² score calculation
def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

# Root Mean Squared Error (RMSE)
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

# Mean Absolute Percentage Error (MAPE)
def mape(y_true, y_pred):
    y_true = np.where(y_true == 0, 1e-6, y_true)  # Avoid division by zero
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Containers for evaluation metrics
tot_r2 = []
tot_rmse = []
tot_mape = []

# Containers for fitted and true capacity loss curves
tot_jl = []
tot_Tcap = []

for num in range(len(tot_cap)):
    # Cycle index within the battery lifetime
    Cycle = np.arange(tot_eol[num])
    
    # Smooth capacity curve using Savitzky–Golay filter
    cap_tmp = savgol_filter(tot_cap[num][:tot_eol[num]], 20, 4)
    
    # Capacity loss calculation
    Qloss = tot_cap[num][0] - cap_tmp

    # Fit the nonlinear degradation model over the entire lifetime
    popt, _ = curve_fit(func, Cycle, Qloss)
    
    # Predict capacity loss over all cycles
    Qloss_pred = func(Cycle, *popt)

    # Store fitted and true capacity loss curves
    tot_jl.append(Qloss_pred)
    tot_Tcap.append(Qloss)

    # Evaluate fitting performance over the full lifetime
    tot_r2.append(r2_score(Qloss, Qloss_pred))
    tot_rmse.append(rmse(Qloss, Qloss_pred))
    tot_mape.append(mape(Qloss, Qloss_pred))

# Output average evaluation metrics
print("Average R²:", np.mean(tot_r2))
print("Average RMSE:", np.mean(tot_rmse))
print("Average MAPE (%):", np.mean(tot_mape))


Average R²: 0.9709241588960776
Average RMSE: 0.009428409213754343
Average MAPE (%): 46.41799666261887


In [5]:
import itertools
from sklearn.model_selection import train_test_split


# Define nonlinear model: y = A * sqrt(x) + B * x^1.5
def model(A, B, x):
    return A * np.sqrt(x) + B * np.power(x, 1.5)


# Recursive Least Squares with regularization and soft constraints
class RecursiveLeastSquares:
    def __init__(self, lambda_factor=0.99, delta=1e3, regularization_factor=0.1):
        # Initialize parameters
        self.lambda_factor = lambda_factor  # Forgetting factor
        self.regularization_factor = regularization_factor  # Regularization strength
        self.theta = np.array([1e-1, 0], dtype=np.float64)  # Initial parameters [A, B]
        self.P = np.eye(2, dtype=np.float64) * delta  # Initial covariance matrix
    
    def update(self, x, y):
        # Feature vector φ_t = [sqrt(x), x^1.5]
        phi_t = np.array([np.sqrt(x), np.power(x, 1.5)], dtype=np.float64)
        
        # Gain matrix K_t
        P_phi = self.P @ phi_t
        gain = P_phi / (self.lambda_factor + phi_t.T @ P_phi)
        
        # Update parameters θ_t
        self.theta += gain * (y - phi_t.T @ self.theta)
        
        # Soft constraint on A to prevent it from becoming negative
        if self.theta[0] < 0:
            self.theta[0] += self.regularization_factor * abs(self.theta[0]) * 4
            if self.theta[0] < 0:
                self.theta[0] = 0
        
        # Update covariance matrix P_t
        self.P = (self.P - np.outer(gain, phi_t.T @ self.P)) / self.lambda_factor
    
    def get_params(self):
        return self.theta  # Return [A, B]


# Batch initialization
def batch_initialize(x_data, y_data, rls, batch_size):
    for i in range(batch_size):
        rls.update(x_data[i], y_data[i])


# Global experiment parameters
window_size = 10  # Rolling window size
increase_length = 12
threshold = 8


# Split temporary dataset into validation and test sets
valid_tot_cap, test_tot_cap, valid_tot_eol, test_tot_eol, valid_tot_tp, test_tot_tp = train_test_split(
    tot_cap, tot_eol, tot_tp, test_size=0.6, random_state=42
)

tot_cap_data = test_tot_cap
tot_tp_data = test_tot_tp
tot_eol_data = test_tot_eol
threshold = threshold * 1e-6
tot_tp_pre = []

# Run experiments on each sample
for num in range(len(tot_cap_data)):
    Cycle = [x for x in range(1000)]
    Qloss = tot_cap_data[num][0] - np.array(tot_cap_data[num])
    y_data = Qloss[:tot_eol_data[num]]
    x_data = np.array(Cycle[:tot_eol_data[num]]) / 1000
    cyc = Cycle[:tot_eol_data[num]]

    # Initialize RLS
    rls = RecursiveLeastSquares(lambda_factor=0.9, delta=1e3)

    # Batch initialization
    batch_initialize(x_data, y_data, rls, batch_size=10)

    # Save history
    A_history = []
    B_history = []
    r2_history = []
    fitted_capacity_history = []

    # Recursive update for each data point
    for t, (x, y) in enumerate(zip(x_data, y_data)):
        rls.update(x, y)
        A, B = rls.get_params()

        A_history.append(A)
        B_history.append(B)

        y_fit_current = model(A, B, x_data[:t + 1])

        fitted_capacity_history.append(y_fit_current[-1])  # Save last fitted value

    t = np.arange(0, len(A_history))  # Time series
    series = pd.Series(A_history)

    # Compute rolling variance
    variance_signal = series.rolling(window=window_size).var()
    moving_avg = np.convolve(variance_signal, np.ones(window_size) / window_size, mode='valid')
    padding = np.full(window_size - 1, np.nan)  # Padding with NaN
    variance_signal = np.concatenate((padding, moving_avg))

    increasing_start_idx = None
    increase_count = 0

    # Detect start index of continuous increase
    for idx in range(80, len(variance_signal)):
        if variance_signal[idx] > variance_signal[idx - 1] and variance_signal[idx] > threshold:
            increase_count += 1
            if increase_count >= increase_length:
                increasing_start_idx = idx - increase_length + 1
                break
        else:
            increase_count = 0  # Reset counter

    # Ensure no out-of-bound access
    if increasing_start_idx is not None:
        increasing_start_time = increasing_start_idx
    else:
        increasing_start_time = len(A_history)

    tot_tp_pre.append(increasing_start_time)


# Evaluate prediction results
prd_er = np.array(tot_tp_data) - np.array(tot_tp_pre)
pos_prd_er = []
pos_tp = []
pos_pr = []
true_positive = []
all_prediction_advance_times = []  # Store all advance times

for i in range(len(prd_er)):
    if tot_tp_pre[i] is not None:  # Skip None values
        time_diff = prd_er[i]  # Prediction lead time
        if time_diff >= 0:
            pos_pr.append(tot_tp_pre[i])
            pos_prd_er.append(time_diff)

        # Count true positives: predicted within 20% of EOL
        if 0 < time_diff <= tot_eol_data[i] * 0.2:
            true_positive.append(tot_tp_pre[i])

# Compute metrics
tp = len(true_positive)
fp = len(pos_pr) - len(true_positive)
tn = 0
fn = len(tot_tp_data) - len(pos_pr)

precision = tp / (tp + fp) if tp + fp != 0 else 0
recall = tp / (tp + fn) if tp + fn != 0 else 0
specificity = tn / (tn + fp) if tn + fp != 0 else 0
f1_score_value = 2 * precision * recall / (precision + recall) if precision + recall != 0 else 0

mean_prediction_advance_time = np.mean(pos_prd_er) if len(pos_prd_er) > 0 else 0
detection_success_rate = len(pos_prd_er) / len(tot_tp_data)

print(f"Warning rate: {detection_success_rate:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Mean prediction advance time (cycles): {mean_prediction_advance_time:.1f}")


Warning rate: 0.8462
Precision: 1.0000
Mean prediction advance time (cycles): 23.6


In [6]:
# Add labels based on prediction performance
labels = []
for i in range(len(tot_tp_data)):
    if prd_er[i] >= 0:
        if tot_tp_pre[i] in true_positive:
            labels.append("Effective warning")
        else:
            labels.append("Ineffective warning")
    else:
        labels.append("Failed warning")

# Create a DataFrame with prediction results
df_results = pd.DataFrame({
    'Warning time': tot_tp_pre,
    'Knee time': tot_tp_data,
    'Label': labels
})

# Count the number of each warning type
effective_count = labels.count("Effective warning")
ineffective_count = labels.count("Ineffective warning")
failed_count = labels.count("Failed warning")

# Print the statistics
print(f"Effective warning count: {effective_count}")
print(f"Ineffective warning count: {ineffective_count}")
print(f"Failed warning count: {failed_count}")


Effective warning count: 11
Ineffective warning count: 0
Failed warning count: 2


In [7]:
# =========================
# Use full dataset (no split)
# =========================
tot_cap_data = tot_cap
tot_tp_data = tot_tp
tot_eol_data = tot_eol

batch_size = 10
eps = 1e-12           # 判定A≈0的数值阈值（可调：1e-12~1e-8）
min_check_cycle = 0   # 可选：比如设80，避免太早误触发；不想限制就用0

tot_tp_pre = []       # 记录每个电池“首次A=0”的cycle；若未触发则为np.nan

for num in range(len(tot_cap_data)):
    # 更通用：Cycle按容量长度来，避免固定1000截断
    cap = np.array(tot_cap_data[num], dtype=np.float64)
    eol = int(tot_eol_data[num])

    Cycle = np.arange(len(cap))
    Qloss = cap[0] - cap

    y_data = Qloss[:eol]
    x_data = (Cycle[:eol] / 1000.0).astype(np.float64)

    # Initialize RLS
    rls = RecursiveLeastSquares(lambda_factor=0.8, delta=1e3)

    # Batch initialization (use first batch_size points once)
    init_n = min(batch_size, len(x_data))
    for i in range(init_n):
        rls.update(x_data[i], y_data[i])

    first_zero_cycle = np.nan

    # Realtime monitoring: start updating from init_n (avoid double-counting)
    for t in range(init_n, len(x_data)):
        rls.update(x_data[t], y_data[t])
        A, B = rls.get_params()

        # record first time A becomes ~0
        if t >= min_check_cycle and A <= eps:
            first_zero_cycle = t
            break

    tot_tp_pre.append(first_zero_cycle)

tot_tp_pre = np.array(tot_tp_pre, dtype=np.float64)
tot_tp_true = np.array(tot_tp_data, dtype=np.float64)

# =========================
# Metrics
# =========================
detected_mask = ~np.isnan(tot_tp_pre)   # 成功监测到A=0
success_rate = np.mean(detected_mask)

# 只在“成功监测到”的样本上算误差（更符合你要的“监测到跳水点后有多准”）
pred = tot_tp_pre[detected_mask]
true = tot_tp_true[detected_mask]

if len(true) > 0:
    # 避免true=0导致MAPE除0（一般tp不会为0，但这里防一下）
    true_safe = np.where(true == 0, 1e-6, true)

    abs_err = np.abs(pred - true)
    mape = np.mean(abs_err / true_safe) * 100.0
    rmse = np.sqrt(np.mean((pred - true) ** 2))
else:
    mape = 0.0
    rmse = 0.0

print(f"SuccessRate:{success_rate:.4f}")
print(f"MAPE(%):{mape:.2f}")
print(f"RMSE(Cycle):{rmse:.2f}")


SuccessRate:0.9524
MAPE(%):11.55
RMSE(Cycle):47.20
